# Cable news chyrons

Associated chart: the chyron ternary and the Griner-swap scrolly

Output files:
`chyrons.json` in `src/data/`.

Input files: `data/chyron_matches.tsv`, `data/chyron_monthly.json` (both harvested from the Internet Archive's Third Eye project).


## Setup

In [1]:
import json, math, re
from collections import Counter, defaultdict
from pathlib import Path

DATA = Path("../data")
OUT = Path("../src/data")

STOP = set("""a an the and or but of in on at to for with by from as is are was were be been
being it its this that these those i you he she they we his her their our your my me him them
us new says said say after amid over vs vs. into out up down off not no what who why how when
where which while will would could should can may might must have has had do does did done
about more most so than then just also only even still back get gets got make makes made
""".split())

# The chyrons are OCR'd, so some "words" are misreads ("qffers"), graphics
# ("b-reerg") or name captions. Common misreads are fixed first; then a word
# only counts if it's in the dictionary or used twice in the news corpus.
OCR_FIX = {
    "wwnba": "wnba", "vnba": "wnba", "rwnba": "wnba", "owa": "iowa", "jowa": "iowa",
    "ndiana": "indiana", "llinois": "illinois", "qffers": "offers", "videq": "video",
    "bullt": "built", "vears": "years", "qut": "out", "plaver": "player", "griney": "griner",
    "brtney": "brittney", "itney": "brittney", "mitney": "brittney", "pbrittney": "brittney",
    "ritey": "brittney", "russan": "russian", "boycoti": "boycott", "quilty": "guilty",
    "broiher": "brother", "democrais": "democrats", "disaprity": "disparity",
    "condenns": "condemns", "awmakers": "lawmakers", "mordqvia": "mordovia",
    "nental": "mental", "admn": "admin", "mccouaghtry": "mccoughtry", "anagel": "angel",
    "echnical": "technical", "siraiegic": "strategic", "bden": "biden", "faul": "foul",
    "fredom": "freedom", "abusurd": "absurd", "quard": "guard", "reponds": "responds",
    "withesses": "witnesses", "trving": "trying", "discriination": "discrimination",
    "playorf": "playoff", "coack": "coach", "appruves": "approves", "activis": "activist",
    "ueffler": "loeffler", "loffe": "loeffler", "loffler": "loeffler", "tloud": "cloud",
    "co-hosi": "co-host", "sponts": "sports", "rpt": "report",
}
# real but not in either list
WORD_EXTRA = {"mrc", "blm", "wnba", "dei", "maga", "nil", "covid", "lgbtq"}

def _news_vocab():
    cache = Path(".cache/news_vocab.json")
    if cache.exists():
        return set(json.loads(cache.read_text()))
    import csv
    csv.field_size_limit(10 ** 9)
    seen = Counter()
    for f in sorted(DATA.glob("*.csv")):
        try:
            for row in csv.DictReader(f.open(encoding="utf-8", errors="ignore")):
                txt = " ".join(row.get(k) or "" for k in ("headline", "title", "body", "text"))
                seen.update(re.findall(r"[a-z][a-z'-]+", txt.lower().replace("’", "'")))
        except Exception:
            continue
    vocab = {w for w, n in seen.items() if n >= 2}
    vocab |= {w.strip().lower() for w in open("/usr/share/dict/words")}
    cache.parent.mkdir(exist_ok=True)
    cache.write_text(json.dumps(sorted(vocab)))
    return vocab

VOCAB = _news_vocab()

def is_word(t):
    t = t.strip("-'")
    if t in VOCAB or t in WORD_EXTRA:
        return True
    # hyphenated words count if their pieces do; short-piece ones like "b-r"
    # (CNN's BREAKING banner) only if the news uses them whole ("op-ed")
    pieces = [p for p in t.split("-") if p]
    if ("-" in t and any(len(p) > 2 for p in pieces) and len(pieces[0]) > 1
            and all(p in VOCAB or len(p) <= 2 for p in pieces)):
        return True
    return any(t.endswith(s) and t[: -len(s)] in VOCAB for s in ("s", "es", "ed", "ing", "ly", "er", "ers"))

def tokenize(text):
    text = re.sub(r"[‘’']s\b", "", text.lower())
    # doesn't -> does
    text = re.sub(r"n[‘’']t\b", "", text)
    toks = (OCR_FIX.get(t, t) for t in re.findall(r"[a-z][a-z-]+", text))
    return [t for t in toks if is_word(t)]

CH_MAP = {"FOXNEWSW": "Fox News", "CNNW": "CNN", "MSNBCW": "MSNBC", "MSNOW": "MSNBC"}
channels = ["Fox News", "CNN", "MSNBC"]

chy = json.loads((DATA / "chyron_monthly.json").read_text())
MO_RE = re.compile(r"^20\d{2}-\d{2}$")
tot = defaultdict(Counter)
mat = defaultdict(Counter)
for raw_ch, label in CH_MAP.items():
    for mo, n in chy.get("totals", {}).get(raw_ch, {}).items():
        if MO_RE.match(mo): tot[label][mo] += n
    for mo, n in chy.get("matches", {}).get(raw_ch, {}).items():
        if MO_RE.match(mo): mat[label][mo] += n

all_months = sorted(set().union(*[set(t) for t in tot.values()]))
def nxt_mo(m):
    y, mm = int(m[:4]), int(m[5:7])
    return f"{y + (mm == 12)}-{(mm % 12) + 1:02d}"
axis, cur = [], all_months[0]
while cur <= all_months[-1]:
    axis.append(cur)
    cur = nxt_mo(cur)
print(f"{axis[0]} → {axis[-1]} · {len(axis)} months")

2017-09 → 2026-10 · 110 months


## Word counts per network

A Third Eye row often bundles several rotating banners; only the SEGMENTS that themselves
match a WNBA term are kept.


In [2]:
CHY_DROP = set("""wnba caitlin clark angel reese brittney griner sophie cunningham
    paige bueckers aja a'ja wilson diana taurasi sue bird candace parker sabrina
    ionescu breanna stewart napheesa collier kelsey plum maya moore skylar diggins
    aliyah boston cameron brink dijonai carrington kamilla cardoso chennedy carter
    indiana fever chicago sky atlanta dream star stars player players""".split()) | STOP
CHY_NAMES = {"caitlin", "clark", "angel", "reese", "brittney", "griner",
             "sophie", "cunningham", "paige", "bueckers", "dijonai",
             "carrington", "diana", "taurasi", "sue", "bird", "candace",
             "parker", "sabrina", "ionescu", "breanna", "stewart",
             "napheesa", "collier", "kelsey", "plum", "maya", "moore",
             "skylar", "diggins", "aliyah", "boston", "cameron", "brink",
             "kamilla", "cardoso", "chennedy", "carter", "wilson"}
# a banner segment counts as WNBA when it contains any of these
WNBA_TERMS = [r"\bwnba\b", "caitlin clark", "angel reese", "brittney griner", r"\bgriner\b",
    "sophie cunningham", "paige bueckers", r"a'?ja wilson", "diana taurasi", "sue bird",
    "candace parker", "sabrina ionescu", "breanna stewart", "napheesa collier",
    "kelsey plum", "maya moore", "skylar diggins", "aliyah boston", "cameron brink",
    "dijonai carrington", "kamilla cardoso", "chennedy carter",
    "indiana fever", "chicago sky", "atlanta dream"]
SEG_RE = re.compile("|".join(WNBA_TERMS), re.I)
PLAYER_NAME_SETS = {
    "Caitlin Clark": {"clark"},
    "Angel Reese": {"reese"},
    "Brittney Griner": {"griner"},
    "Sophie Cunningham": {"cunningham"},
}
ERAS = ("pre", "post")  # before 2023 / 2023 onward

ex = defaultdict(dict)
words = defaultdict(Counter)
word_ex = defaultdict(dict)
adj_words = defaultdict(Counter)
adj_ex = defaultdict(dict)
pl_cooc = {p: defaultdict(Counter) for p in PLAYER_NAME_SETS}
pl_totals = {p: Counter() for p in PLAYER_NAME_SETS}
pl_cooc_era = {e: {p: defaultdict(Counter) for p in PLAYER_NAME_SETS} for e in ERAS}
pl_totals_era = {e: {p: Counter() for p in PLAYER_NAME_SETS} for e in ERAS}
words_era = {e: defaultdict(Counter) for e in ERAS}
pl_monthly = {p: defaultdict(Counter) for p in PLAYER_NAME_SETS}
mat_strict = defaultdict(Counter)
# the ternary keeps the players' names (so "clark" can be plotted); everything
# else about the word lists still leaves them out
TERN_DROP = (CHY_DROP - CHY_NAMES) | {"aja", "a'ja"}
words_tern = defaultdict(Counter)
word_ex_tern = defaultdict(dict)
# the same counts by year, for the triangle's timeline: [year][channel][word]
words_tern_year = defaultdict(lambda: defaultdict(Counter))
def clean_ex(text):
    """Tidy an OCR'd banner for display: decode escape sequences that came
    through literally, straighten quotes, drop stray symbols, collapse the
    ". ." fragments, and cut the ticker/clock scraps."""
    text = re.sub(r"\\u([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), text)
    text = text.replace('\\"', '"').replace("\\n", " / ").replace("\\", " ")
    for a, b in {"“": '"', "”": '"', "‘": "'", "’": "'", "—": "-",
                 "–": "-", "ﬁ": "fi", "ﬂ": "fl"}.items():
        text = text.replace(a, b)
    text = re.sub(r"\|?\s*\d{1,2}(?::\d\d)?\s*[AP]M\s*ET:?", " ", text, flags=re.I)  # | 12PMET:
    text = re.sub(r"\b[1IL]IVE\s*>.*?(?=/|$)", " ", text)                          # 1IVE > 7:5Q°Fm …
    text = re.sub(r"[^A-Za-z0-9 .,:;!?'\"\-&$%/()]", " ", text)                     # anything else odd
    text = re.sub(r"(?:\s*\.\s*){2,}", ". ", text)                                  # ". ." fragments
    text = re.sub(r"\s+", " ", text).strip(" .,:;/-|")
    return text

# example chyrons: misreads fixed, junk trimmed, and the cleanest one kept
_FIX_RE = re.compile(r"\b(" + "|".join(map(re.escape, OCR_FIX)) + r")\b", re.I)

def _repair(word):
    """OCR reads T as I' ("AFI'ER"); swap it back if that makes a word"""
    if "'" not in word or is_word(re.sub(r"[^a-z'-]", "", word.lower())):
        return word
    for a, b in (("I'I'", "TT"), ("'I'I", "TT"), ("I'I", "TT"), ("I'", "T"), ("'I", "T")):
        fixed = word.replace(a, b)
        if fixed != word and is_word(re.sub(r"[^a-z'-]", "", fixed.lower())):
            return fixed
    return word

def tidy_ex(text, keep=()):
    text = _FIX_RE.sub(lambda m: OCR_FIX[m.group(1).lower()].upper(), clean_ex(text))
    # still in the export's quoting: a quote around the whole thing, and
    # doubled ones inside (""ZOO ANIMAL"")
    text = text.strip()
    if len(text) > 1 and text[0] == '"' and text[-1] == '"':
        text = text[1:-1]
    text = re.sub(r'"{2,}', '"', text)
    # an odd one out, alone at the start ('" GRINER: …'), is a stray
    if text.count('"') % 2 and text.startswith('" '):
        text = text[2:]
    parts = [_repair(p) for p in text.split(" ")]
    junk = lambda p: bool(re.search(r"[A-Za-z]", p)) and not any(
        is_word(w) for w in re.findall(r"[a-z][a-z'-]+", p.lower()))
    # misreads with stray capitals ("EEgd") go wherever they are
    odd_case = lambda p: bool(re.search(r"[A-Z]{2,}[a-z]|[a-z][A-Z]", p))
    parts = [p for p in parts if not (junk(p) and odd_case(p))]
    while parts and junk(parts[0]):
        parts.pop(0)
    while parts and junk(parts[-1]):
        parts.pop()
    text = " ".join(parts)
    # a scrap of misread text before (or after) a dash, mostly junk even if
    # one bit reads as a word: "AN YT R N. - WNBA STAR CAITLIN CLARK …"
    # (never the piece holding the word the example is for)
    halves = text.split(" - ")
    holds = lambda s: any(str(k).lower() in s.lower() for k in keep if k)
    scrap = lambda s: bool(s.split()) and not holds(s) and sum(junk(p) for p in s.split()) * 2 >= len(s.split())
    if len(halves) > 1 and scrap(halves[0]):
        halves = halves[1:]
    if len(halves) > 1 and scrap(halves[-1]):
        halves = halves[:-1]
    text = " - ".join(halves)
    # a misread tail after the last full stop of an all-caps chyron: short,
    # and holding a lowercase scrap or mostly junk ("… TAYLOR. kl TMC LIN")
    head, dot, tail = text.rpartition(". ")
    letters = [ch for ch in head if ch.isalpha()]
    caps = letters and sum(ch.isupper() for ch in letters) >= 0.8 * len(letters)
    if (dot and caps and len(tail.split()) <= 4 and not holds(tail)
            and (re.search(r"(?<![A-Za-z])[a-z]{1,3}(?![A-Za-z])", tail)
                 or sum(junk(p) for p in tail.split()) * 2 >= len(tail.split()))):
        text = head
    # a cut-off bracket at the end ("… (TR")
    text = re.sub(r"\s*\([^)]{0,4}$", "", text)
    return text.strip(" .,:;/-|")

def non_words(text):
    # misread words, and stray lowercase scraps in an all-caps chyron ("CO-HOST. i BIDEN", "kl")
    scraps = re.findall(r"(?<![A-Za-z])[a-z]{1,3}(?![A-Za-z])", text) if text.upper() != text else []
    stray = sum(len(s) == 1 or not is_word(s) for s in scraps)
    return sum(not is_word(w) for w in re.findall(r"[a-z][a-z'-]+", text.lower())) + stray

_ex_score = {}
def find_word(text, words):
    # where the word the example is for first shows up (None if it doesn't)
    hits = [m.start() for w in words if w
            for m in [re.search(r"(?<![A-Za-z])" + re.escape(str(w)), text, re.I)] if m]
    return min(hits) if hits else None

def clip(text, n, words=()):
    if len(text) <= n:
        return text
    # a word that falls past the cut: the piece is taken around it instead
    at = find_word(text, words)
    start = 0 if at is None or at + 20 <= n else text.rfind(" ", 0, at - n // 3) + 1
    piece = text[start:start + n]
    if start + n < len(text):
        piece = piece.rsplit(" ", 1)[0].rstrip(" .,:;/-|") + "…"
    return ("…" + piece.lstrip(" .,:;/-|")) if start else piece

def keep_ex(store, key, label, raw, n=150):
    text = clip(tidy_ex(raw, keep=(key, label)), n, words=(key, label))
    # an example that shows its word beats one that doesn't
    score = non_words(text) + (0 if find_word(text, (key, label)) is not None else 100)
    k = (id(store), key, label)
    if label not in store[key] or score < _ex_score.get(k, 1e9):
        store[key][label] = text
        _ex_score[k] = score

kept_of = lambda toks: [t for t in toks
                        if (t not in CHY_NAMES and t not in CHY_DROP and len(t) > 2
                            and not t.endswith("-") and t not in {"com", "www", "cnn", "fox", "msnbc"})]

for line in open(DATA / "chyron_matches.tsv"):
    parts = line.rstrip("\n").split("\t")
    if len(parts) < 5 or parts[0].startswith("date_"):
        continue
    label = CH_MAP.get(parts[1])
    if not label:
        continue
    mo = parts[0][:7]
    text = parts[4].replace("\\n", " / ")
    # advertisement chyrons (e.g. bedding ads citing a former WNBA player)
    if re.search(r"sheex|marciniak|promo code|\.com/", text, re.I):
        continue
    segs = [s.strip() for s in re.split(r"\s*/\s*", text) if s.strip()]
    wnba_segs = [s for s in segs if SEG_RE.search(s)]
    if not wnba_segs:
        continue
    mat_strict[label][mo] += 1
    text_w = " / ".join(wnba_segs)
    keep_ex(ex, label, mo, text_w, 160)
    era = "pre" if mo < "2023" else "post"
    for t in set(tokenize(text_w)):
        if len(t) > 2 and not t.endswith("-") and t not in {"com", "www", "cnn", "fox", "msnbc"}:
            if t not in CHY_DROP:
                words[label][t] += 1
                words_era[era][label][t] += 1
                keep_ex(word_ex, t, label, text_w)
    # the ternary counts every use of a word, not just the banners it appears in
    for t in tokenize(text_w):
        if (len(t) > 2 and not t.endswith("-") and t not in {"com", "www", "cnn", "fox", "msnbc"}
                and t not in TERN_DROP):
            words_tern[label][t] += 1
            words_tern_year[mo[:4]][label][t] += 1
            keep_ex(word_ex_tern, t, label, text_w)
    hit_players = set()
    for s in wnba_segs:
        s_toks = set(tokenize(s))
        kept = kept_of(s_toks)
        if any(t in CHY_NAMES for t in s_toks):
            for t in kept:
                adj_words[label][t] += 1
                keep_ex(adj_ex, label, t, s)
        for player, names in PLAYER_NAME_SETS.items():
            if names & s_toks:
                hit_players.add(player)
                for t in kept:
                    pl_cooc[player][label][t] += 1
                    pl_cooc_era[era][player][label][t] += 1
    for player in hit_players:
        pl_totals[player][label] += 1
        pl_totals_era[era][player][label] += 1
        pl_monthly[player][label][mo] += 1

# the monthly matched series uses the stricter segment-level counts
if sum(sum(c.values()) for c in mat_strict.values()):
    mat = mat_strict
print({c: sum(mat[c].values()) for c in channels})

{'Fox News': 1713, 'CNN': 3695, 'MSNBC': 2319}


## The ternary and the output

Each word's share across the three networks.


In [3]:
# names included here (words_tern), unlike the other word lists
ch_word_totals = {c: sum(words_tern[c].values()) for c in channels}
all_w = Counter()
for c in channels:
    all_w.update(words_tern[c])
tern_terms = []
# every word that appears on a WNBA chyron, no floor
for term, total in all_w.most_common(50000):
    if total < 1:
        break
    raw_c = [words_tern[c][term] for c in channels]
    rates = [n / max(ch_word_totals[c], 1) for n, c in zip(raw_c, channels)]
    ssum = sum(rates)
    if ssum == 0:
        continue
    shares = [round(r / ssum, 4) for r in rates]
    dom = channels[shares.index(max(shares))]
    ex_ch = dom if word_ex_tern[term].get(dom) else next(iter(word_ex_tern[term]), "")
    tern_terms.append({"term": term, "total": total, "counts": raw_c,
                       "shares": shares, "ex_ch": ex_ch,
                       "ex": word_ex_tern[term].get(ex_ch, "")})

payload = {
    "months": axis,
    "channels": channels,
    "series": {c: {"matches": [mat[c].get(m, 0) for m in axis],
                   "totals": [tot[c].get(m, 0) for m in axis],
                   "ex": {m: ex[c].get(m, "") for m in axis if ex[c].get(m)}}
               for c in channels},
    "words": {c: [{"w": w, "c": n} for w, n in words[c].most_common(18)] for c in channels},
    "cooc": {c: (lambda ch: [
        {"w": w, "c": n,
         "wt": round(n * max(0.0, math.log(
             (n / max(sum(adj_words[ch].values()), 1)) /
             max(words[ch].get(w, n) / max(sum(words[ch].values()), 1), 1e-9))), 2),
         "ex": adj_ex[ch].get(w, "")}
        for w, n in adj_words[ch].most_common(200) if n >= 3
    ])(c) for c in channels},
    "ternary": tern_terms,
    "player_cooc": {p: {
        c: [{"w": w, "c": n,
             # how much more often the word appears in this player's chyrons
             # than in the channel's other WNBA chyrons
             "wt": round(n * max(0.0, math.log(
                 (n / max(sum(pl_cooc[p][c].values()), 1)) /
                 max((words[c].get(w, n) - n + 0.5) /
                     max(sum(words[c].values()) - sum(pl_cooc[p][c].values()), 1), 1e-9))), 2)}
            for w, n in pl_cooc[p][c].most_common(200) if n >= 3]
        for c in channels}
        for p in pl_cooc},
    "player_chyrons": {p: {c: pl_totals[p][c] for c in channels} for p in pl_totals},
    "player_cooc_era": {e: {p: {
        c: [{"w": w, "c": n,
             "wt": round(n * max(0.0, math.log(
                 (n / max(sum(pl_cooc_era[e][p][c].values()), 1)) /
                 max((words_era[e][c].get(w, n) - n + 0.5) /
                     max(sum(words_era[e][c].values()) - sum(pl_cooc_era[e][p][c].values()), 1), 1e-9))), 2)}
            for w, n in pl_cooc_era[e][p][c].most_common(200) if n >= 3]
        for c in channels}
        for p in pl_cooc_era[e]} for e in ERAS},
    "player_chyrons_era": {e: {p: {c: pl_totals_era[e][p][c] for c in channels}
                               for p in pl_totals_era[e]} for e in ERAS},
    "player_monthly": {p: {c: [pl_monthly[p][c].get(m, 0) for m in axis] for c in channels}
                       for p in pl_monthly},
}

#### Overwrites `src/data/chyrons.json`

In [4]:
(OUT / "chyrons.json").write_text(json.dumps(payload))

# the triangle over time: each channel's word total and each word's counts by
# year, so the site can place the words for any year (or any run of years)
years_t = sorted(words_tern_year)
year_totals = {y: [sum(words_tern_year[y][c].values()) for c in channels] for y in years_t}
by_term = {}
for y in years_t:
    for c_i, c in enumerate(channels):
        for term, n in words_tern_year[y][c].items():
            if all_w[term] < 2:
                continue
            by_term.setdefault(term, {}).setdefault(y, [0] * len(channels))[c_i] = n
(OUT / "chyron_ternary_years.json").write_text(json.dumps(
    {"years": years_t, "channels": channels, "totals": year_totals, "terms": by_term}))
print(f"{len(by_term)} words by year -> chyron_ternary_years.json")
griner = {t["term"]: t["shares"] for t in tern_terms if t["term"] in ("viktor", "bout", "merchant", "death")}
print(f"{len(tern_terms)} ternary terms · wrote chyrons.json")
griner

3236 words by year -> chyron_ternary_years.json
4117 ternary terms · wrote chyrons.json


{'bout': [0.0359, 0.1947, 0.7694],
 'viktor': [0.0, 0.186, 0.814],
 'death': [0.687, 0.226, 0.0871],
 'merchant': [0.7008, 0.2104, 0.0888]}

## Spot check

In [5]:
import random

print("— random ternary words, with a real chyron each")
for t in random.sample(tern_terms, 5):
    lean = channels[t["shares"].index(max(t["shares"]))]
    print(f"   {t['term']!r} leans {lean} {t['shares']}")
    if t["ex"]: print(f"      “{t['ex'][:90]}”")

player = random.choice(list(pl_cooc))
ch = random.choice(channels)
words_here = pl_cooc[player][ch].most_common(30)
print(f"\n— random {ch} words sharing a banner with {player}")
for w, n in random.sample(words_here, min(4, len(words_here))):
    print(f"   {w} ({n})  “{adj_ex[ch].get(w, '')[:70]}”")

— random ternary words, with a real chyron each
   'attended' leans MSNBC [0.0, 0.2803, 0.7197]
      “VICE PRESIDENT HARRIS AMONG NEARLY 11K. l WHO ATTENDED BRITTNEY GRINER'S FIRST OFFICIAL GA”
   'free' leans MSNBC [0.2752, 0.1658, 0.5589]
      “WNBA STAR HELPS FREE MAN FROM PRISON AFTER 23 YEARS”
   'child' leans MSNBC [0.0, 0.1149, 0.8851]
      “…GRINER / BRITTNEY GRINER ON WHAT SHE'D TELL HER FUTURE CHILD:. "HOLD YOUR HEAD HIGH & KEE”
   'became' leans CNN [0.0, 1.0, 0.0]
      “sPoRTS CAITLIN CLARK PICKED NO. 1 BY INDIANA FEVER IN WNBA DRAFT. MORNING 22-year-old beca”
   'organization' leans CNN [0.0, 0.609, 0.391]
      “sPORTS USA BASKETBALL ANNOUNCES OFFICIAL WOMEN'S OLYMPIC ROSTER. MORNING Organization says”

— random CNN words sharing a banner with Brittney Griner
   guilty (127)  “"WNBA STAR BRITTNEY GRINER PLEADS GUILTY TO DRUG CHARGES IN RUSSIA. Ky”
   home (218)  “WNBA PLAYERS UNION PRES. ON BRITTNEY GRINER: "WE WANT HER HOME"”
   swap (557)  “DETAINED AMERICAN'S BROTH